# Getting our hands dirty: Data Cleaning

Before we can find any cool insights, we need to clean up the mess. The raw data from Inside Airbnb is a bit chaotic, so we are going to go through it in three main steps to make sure it is ready for our analysis.

In [7]:
import pandas as pd

# Load the raw dataset
file_path = '../data/raw/airbnb_listing.csv'
df = pd.read_csv(file_path, low_memory=False)
print(f'Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns')

Loaded dataset: 42904 rows × 38 columns


## Step 1: Cutting out the noise

First things first, we don't need columns like  or . These are just system artifacts from when the data was collected and won't help us understand the market at all.

In [8]:
pass1_cols = [
    # Scraping / system metadata
    'scrape_id', 'last_scraped', 'source', 'calendar_updated', 'calendar_last_scraped',
    # Internal IDs & URLs
    'id', 'listing_url', 'host_id', 'host_url',
    # Image URLs
    'picture_url', 'host_thumbnail_url', 'host_picture_url',
    # Overly-granular booking rules (we keep minimum_nights & maximum_nights)
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights',
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    # Redundant / highly-missing geography
    'neighbourhood_group_cleansed', 'host_neighbourhood', 'neighbourhood',
    # Redundant calculated host listing counts
    'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
    # Legal
    'license'
]

existing = [c for c in pass1_cols if c in df.columns]
df = df.drop(columns=existing)
print(f'Pass 1: Dropped {len(existing)} columns → {df.shape[1]} columns remaining')

Pass 1: Dropped 0 columns → 38 columns remaining


## Step 2: Focus on the Business

Since we are looking at this from an investment perspective, we are going to drop things like host bios and long descriptions. We want to focus on the "hard" data like price, location, and property type.

In [9]:
pass2_cols = [
    # Personal host profile — not useful for business location/pricing decisions
    'host_response_rate',
    'host_total_listings_count',
    'host_has_profile_pic',
    'host_verifications',
    'host_name',
    'host_about',             # 45% missing, free-text bio
    'host_location',          # 23% missing, redundant with listing lat/lon
    'host_identity_verified',
    # Free-text fields — cannot be used in statistical/quantitative analysis
    'description',
    'name',                   # listing title
    'neighborhood_overview',  # 48.5% missing, free text
    # Redundant date / availability fields
    'first_review',           # date of first review, not strategic
    'last_review',            # date of last review, not strategic
    'has_availability',       # boolean flag, redundant with availability_30/60/90/365
    'bathrooms_text',         # text version of bathrooms, redundant with numeric 'bathrooms'
    'number_of_reviews_l30d', # too short-term for business planning
    'availability_eoy',       # end-of-year calendar artifact
]

existing = [c for c in pass2_cols if c in df.columns]
df = df.drop(columns=existing)
print(f'Pass 2: Dropped {len(existing)} columns → {df.shape[1]} columns remaining')

Pass 2: Dropped 2 columns → 36 columns remaining


## Pass 3 — Missing Values & Currency Formatting
To ensure our business analysis doesn't break on missing data:
- Remove currency symbols (`$`, `,`) from `price` to temporarily convert it to a float.
- Fill all numeric columns' missing values (`NaN`) with their **median**.
- Re-add the `$` symbol back to `price` for presentation formatting.

In [10]:
# Step 3: Fixing the numbers and filling missing values
import pandas as pd
import numpy as np

# 1. Clean columns with symbols ($, %)
if "price" in df.columns:
    df["price"] = df["price"].astype(str).str.replace(r"[\$,]", "", regex=True)

if "host_acceptance_rate" in df.columns:
    df["host_acceptance_rate"] = df["host_acceptance_rate"].astype(str).str.replace("%", "")

if "host_response_rate" in df.columns:
    df["host_response_rate"] = df["host_response_rate"].astype(str).str.replace("%", "")

# 2. Force all columns that should be numeric to numeric
cols_to_force = [
    "price", "accommodates", "bedrooms", "beds", "bathrooms", 
    "minimum_nights", "maximum_nights", "availability_30", 
    "availability_60", "availability_90", "availability_365", 
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_ly",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_checkin",
    "review_scores_communication", "review_scores_location", "review_scores_value",
    "review_scores_accuracy", "host_acceptance_rate", "host_listings_count",
    "estimated_occupancy_l365d", "estimated_revenue_l365d", "reviews_per_month"
]

for col in cols_to_force:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Fill all missing values in numeric columns with their median
numeric_cols = df.select_dtypes(include=["number"]).columns
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: filled with median = {median_val}")

# 4. Handle Categorical missing values
cat_cols = df.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

print("--- Cleaning Summary ---")
print(f"Final columns: {df.shape[1]}")
print(df.dtypes.value_counts())


  review_scores_accuracy: filled with median = 4.9
  host_acceptance_rate: filled with median = 100.0
  availability_90: filled with median = 54.0
  availability_60: filled with median = 28.0
  bedrooms: filled with median = 1.0
  review_scores_cleanliness: filled with median = 4.87
  reviews_per_month: filled with median = 0.66
  host_listings_count: filled with median = 3.0
  longitude: filled with median = 23.725931850000002
  bathrooms: filled with median = 1.0
  review_scores_rating: filled with median = 4.87
  estimated_occupancy_l365d: filled with median = 18.0
  estimated_revenue_l365d: filled with median = 4620.0
  number_of_reviews_ly: filled with median = 2.0
  review_scores_checkin: filled with median = 4.94
  accommodates: filled with median = 3.0
  latitude: filled with median = 37.978036613631026
  maximum_nights: filled with median = 365.0
  minimum_nights: filled with median = 2.0
  number_of_reviews: filled with median = 13.0
  review_scores_location: filled with medi

## Final Dataset Summary
The remaining columns are all directly relevant to answering business questions:
- **Location**: `neighbourhood_cleansed`, `latitude`, `longitude`
- **Pricing & Revenue**: `price`, `estimated_revenue_l365d`
- **Property Details**: `property_type`, `room_type`, `bedrooms`, `beds`, `bathrooms`, `accommodates`, `amenities`
- **Demand Signals**: `estimated_occupancy_l365d`, `availability_30/60/90/365`, `reviews_per_month`, `number_of_reviews`
- **Quality Scores**: `review_scores_rating`, `review_scores_cleanliness`, `review_scores_location`, etc.
- **Host Competitiveness**: `host_is_superhost`, `host_acceptance_rate`, `instant_bookable`

In [11]:
# Display all remaining columns
print(f'Final dataset: {df.shape[0]} rows × {df.shape[1]} columns\n')
print('Remaining columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

Final dataset: 42904 rows × 36 columns

Remaining columns:
   1. review_scores_accuracy
   2. host_acceptance_rate
   3. availability_90
   4. neighbourhood_cleansed
   5. availability_60
   6. bedrooms
   7. review_scores_cleanliness
   8. host_response_time
   9. reviews_per_month
  10. amenities
  11. host_listings_count
  12. longitude
  13. room_type
  14. bathrooms
  15. review_scores_rating
  16. estimated_occupancy_l365d
  17. estimated_revenue_l365d
  18. property_type
  19. number_of_reviews_ly
  20. review_scores_checkin
  21. accommodates
  22. latitude
  23. maximum_nights
  24. instant_bookable
  25. host_is_superhost
  26. minimum_nights
  27. host_since
  28. number_of_reviews
  29. review_scores_location
  30. price
  31. review_scores_communication
  32. number_of_reviews_ltm
  33. review_scores_value
  34. availability_30
  35. beds
  36. availability_365


In [12]:
# Save the cleaned dataset
save_path = '../data/processed/cleaned_airbnb.csv'
df.to_csv(save_path, index=False)
print(f'\nCleaned dataset saved to {save_path}')


Cleaned dataset saved to ../data/processed/cleaned_airbnb.csv


In [ ]:
import pandas as pd

df = pd.read_csv("SECTIONA_G5_SmartStay_Analytics/data/processed/cleaned_airbnb.csv")

df.shape
df.info()
df.isnull().sum().sort_values(ascending=False)